# 028 — Modelos ocultos de Markov

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia en 5 minutos

**HMM** = proceso de Markov sobre estados **ocultos** `Xₜ` que solo se observan indirectamente vía **emisiones** `Eₜ`. Tres componentes:

```text
π  : distribución inicial P(X₀)
A  : matriz de transición  P(Xₜ | Xₜ₋₁)   (supuesto de Markov de 1er orden)
B  : modelo de emisión     P(Eₜ | Xₜ)     (la observación solo depende del estado actual)
```

**Los problemas canónicos:**

- **Filtrado (forward):** `P(Xₜ | e₁:ₜ)` — creencia actual. Recurrencia en dos pasos:
  `predicción: b'(x) = Σ_x' P(x|x')·b(x')` y luego `corrección: b(x) ∝ P(eₜ|x)·b'(x)`.
- **Decodificación (Viterbi):** la *secuencia* de estados más probable — igual que forward pero con `max` en lugar de `Σ`, guardando punteros hacia atrás.
- **Suavizado (forward-backward):** `P(Xₖ | e₁:ₜ)` con `k < t` — revisar el pasado con evidencia futura.

Costo por paso: `O(|S|²)` — lineal en la longitud de la secuencia gracias a Markov.


### Mini ejemplo

Estados `{lluvia, sol}` con `P(lluvia_t | lluvia_{t-1}) = 0.7`, `P(lluvia_t | sol_{t-1}) = 0.3`; emisión `paraguas` con `P(par | lluvia) = 0.9`, `P(par | sol) = 0.2`. Partiendo de creencia uniforme y viendo un paraguas: predicción `(0.5, 0.5)`, corrección `∝ (0.9·0.5, 0.2·0.5) = (0.45, 0.10)` → normalizado `(0.818, 0.182)`. El laboratorio `probability` sigue esta misma lógica de actualización de creencia con evidencia ruidosa.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("probability", seed=28)
show(result)


## Reflexión

1. ¿Qué par de supuestos de independencia hacen que el filtrado cueste `O(|S|²)` por paso en lugar de crecer con la historia completa?
2. ¿Por qué la secuencia de Viterbi puede diferir de la secuencia de estados individualmente más probables paso a paso?
3. Si el modelo de emisión fuera perfecto (`P(e|x)` determinista y distinto por estado), ¿qué queda del problema? ¿Y si la transición fuera la identidad?
